# Tetra — Ternary Transformer: train a ~100M model on a free GPU

This notebook trains a **Ternary Transformer** (`-1, 0, +1` weights, STE) on TinyStories
using the free GPU on **Kaggle** (T4/P100) or **Google Colab** (T4). It works on both
— cells auto-detect the environment.

**What happens:**
1. Clone the [Tetra repo](https://github.com/noverisp3/tetra)
2. Install dependencies
3. Auto-download TinyStories (~535M tokens) and tokenize with the custom BPE
4. Train the base model with STE + ternary weights in **float16 on CUDA**
5. Export to the 2-bit C++ binary + quick inference to see it work

**Runtime:** use GPU accelerator (T4, 16 GB is plenty).

> Tips: Edit `PRESET` and `STEPS` in the config cell. `--preset 500m` is ~100M params
> (hidden 2560 × 6 layers + 21M embedding on vocab 8192). Colab has a ~12h session
> limit and Kaggle ~12h too — a 100M model on T4 runs at a few minutes per 1000 steps.

In [ ]:
# --- Cell 1: detect environment + clone repo ---
import os, sys, subprocess, platform, shutil

IS_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
IS_COLAB = "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ
print(f"Environment: {'Kaggle' if IS_KAGGLE else 'Colab' if IS_COLAB else 'local'}")

REPO = "https://github.com/noverisp3/tetra.git"
if not os.path.isdir("tetra"):
    !git clone --depth 1 {REPO}
os.chdir("tetra")
print("Repo at:", os.getcwd())

In [ ]:
# --- Cell 2: install dependencies ---
# Kaggle/Colab already ship torch; this just fills the gaps.
!pip install -q --no-cache-dir numpy tqdm transformers tokenizers datasets requests psutil matplotlib
import torch
print("torch", torch.__version__, "| cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Configure the run

Choose a preset and step budget. Defaults are tuned for a T4 16 GB free session:

| preset | hidden × layers | approx params | time on T4 (1000 steps) |
|---|---|---|---|
| `medium` | 512 × 12 | ~30M | ~2–3 min |
| `large` | 768 × 12 | ~50M | ~4–6 min |
| `500m` | 2560 × 6 | **~100M** | ~10–15 min |

float16 is fast and numerically fine on CUDA (the repo uses GradScaler + FP32 attention on CUDA).

In [ ]:
# --- Cell 3: training configuration ---
PRESET = "500m"      # "tiny" / "medium" / "large" / "500m"
STEPS  = 5000        # training steps
BATCH  = 8           # batch size (larger on GPU is fine; 8 keeps memory low)
GRAD_ACCUM = 4       # gradient accumulation (effective batch = BATCH * GRAD_ACCUM = 32)
BLOCK  = 128         # sequence length (repo default)
LR     = 1e-3        # learning rate (repo default for ternary)

# Logit scale: 1/sqrt(hidden_dim) calibrates initial CE to ~ln(vocab) and
# prevents fp16 overflow on large models (loss ~190 without it on 500m).
# Computed from the preset hidden dim below.
import math
_HIDDEN = {"tiny": 256, "medium": 512, "large": 768, "500m": 2560}[PRESET]
LOGIT_SCALE = round(1.0 / math.sqrt(_HIDDEN), 4)

# Save final model as "checkpoints/checkpoint_final.pt" for easy download
SAVE_DIR = "checkpoints"

print(f"Preset {PRESET} | steps {STEPS} | batch {BATCH}x{GRAD_ACCUM} | block {BLOCK} | lr {LR} | logit_scale {LOGIT_SCALE}")

## 3. Train

This downloads TinyStories (V2 GPT-4, ~1 GB), tokenizes with the custom BPE, then trains.
The first run does the download + tokenize (a few minutes); subsequent runs reuse the cache.

In [ ]:
# --- Cell 4: train on GPU ---
!python train.py \
    --preset {PRESET} \
    --steps {STEPS} \
    --batch-size {BATCH} \
    --grad-accum {GRAD_ACCUM} \
    --block-size {BLOCK} \
    --lr {LR} \
    --logit-scale {LOGIT_SCALE} \
    --dtype float16 \
    --device cuda \
    --save-dir {SAVE_DIR} \
    --save-best \
    --graph

## 4. Export + test

Export the trained checkpoint to the **2-bit ternary binary** (the `.bin` the C++ runtime and
the self-learning `--no-mul` rule consume), then generate a short story with the Python
runtime to confirm the model works.

In [ ]:
# --- Cell 5: export to binary + self-learning metadata ---
import glob

ckpts = sorted(glob.glob(f"{SAVE_DIR}/checkpoint_*.pt"))
ckpts = [c for c in ckpts if "best" not in c]
print("Checkpoints:", ckpts)

# Prefer checkpoint_best.pt if it exists (lowest val loss), else the last one
best = glob.glob(f"{SAVE_DIR}/checkpoint_best.pt")
SOURCE = best[0] if best else ckpts[-1]
print("Using:", SOURCE)

!python inference/export_model.py {SOURCE} -o tetra_100m.bin --self-learning --verify
print("Exported tetra_100m.bin")
!ls -la tetra_100m.bin

In [ ]:
# --- Cell 6: generate a story with the trained model ---
!python inference/run_inference.py tetra_100m.bin "Once upon a time, a little bear" \
    --max-tokens 120 --temp 0.8 --top-k 50 --top-p 0.9 --repeat-penalty 1.1

## 5. On-device continual learning (the differentiator)

Tetra's point of difference: the exported 2-bit model **keeps learning on-device with a
matmul-free rule** (`--no-mul`, Exp 19) — no GPU, no backprop. On this free-GPU session
you can watch the C++/Python self-learning feed on a raw token stream. (The Python path is
demonstrative; the full C++ runtime is in `inference/selflearn.cpp`, built via `build.sh`.)

In [ ]:
# --- Cell 7 (optional): prepare a small token stream for self-learning ---
import numpy as np, json

# Reuse the training cache: take the first 50k tokens of tinystories.bin
bin_path = "data/tinystories.bin"
if os.path.exists(bin_path):
    toks = np.memmap(bin_path, dtype=np.uint16, mode="r")
    head = toks[:50000].astype(np.uint32)
    head.tofile("stream50k.bin")
    print("Wrote stream50k.bin (50k tokens)")
else:
    print("No tinystories.bin cache found; skip self-learning demo.")

## Downloading results

- **Checkpoint:** `checkpoints/checkpoint_best.pt` (PyTorch, for further fine-tuning)
- **Binary:** `tetra_100m.bin` (2-bit ternary, ~the model itself)
- **Plot:** `checkpoints/loss_plot.png` (training curve)

On **Colab**, run the cell below (Files → Download). On **Kaggle**, the files appear in
`/kaggle/working` and are saved as dataset output versions when you save the notebook.

In [ ]:
# --- Cell 8: compress results for download (Colab) ---
if not IS_KAGGLE:
    !mkdir -p /content/download
    !cp tetra_100m.bin /content/download/ 2>/dev/null || true
    !cp {SOURCE} /content/download/ 2>/dev/null || true
    !cp checkpoints/loss_plot.png /content/download/ 2>/dev/null || true
    !cd /content && tar czf /content/tetra_results.tar.gz download 2>/dev/null || true
    print("Download /content/tetra_results.tar.gz")
else:
    print("On Kaggle, results are already in /kaggle/working — use 'Save version'.")